# Playback Event Analytics

Interactive notebook for exploring playback events stored in Apache Iceberg.

## Setup
First, let's create our Spark session with Iceberg configuration.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime, timedelta

spark = (SparkSession.builder
    .appName("PlaybackAnalytics")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "rest")
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.iceberg.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", "iceberg")
    .getOrCreate())

print(f"Spark version: {spark.version}")
print("Session created successfully!")

## Explore Available Tables

In [ ]:
# List all tables in the warehouse namespace
spark.sql("SHOW TABLES IN iceberg.warehouse").show()

## Event Statistics

In [ ]:
# Get basic event statistics
spark.sql("""
    SELECT 
        COUNT(*) as total_events,
        COUNT(DISTINCT user_id) as unique_users,
        COUNT(DISTINCT session_id) as unique_sessions,
        COUNT(DISTINCT content_id) as unique_content,
        MIN(event_timestamp) as first_event,
        MAX(event_timestamp) as last_event
    FROM iceberg.warehouse.playback_events
""").show()

In [ ]:
# Event type distribution
event_types = spark.sql("""
    SELECT 
        event_type,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM iceberg.warehouse.playback_events
    GROUP BY event_type
    ORDER BY count DESC
""")
event_types.show()

## Device Analytics

In [ ]:
# Device type distribution
spark.sql("""
    SELECT 
        device_type,
        COUNT(*) as events,
        COUNT(DISTINCT user_id) as users,
        ROUND(AVG(bitrate_kbps), 0) as avg_bitrate
    FROM iceberg.warehouse.playback_events
    GROUP BY device_type
    ORDER BY events DESC
""").show()

## Hourly Traffic Pattern

In [ ]:
# Hourly event distribution
hourly_events = spark.sql("""
    SELECT 
        event_hour,
        COUNT(*) as events,
        COUNT(DISTINCT user_id) as unique_users
    FROM iceberg.warehouse.playback_events
    GROUP BY event_hour
    ORDER BY event_hour
""").toPandas()

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(hourly_events['event_hour'], hourly_events['events'], alpha=0.7, label='Events')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Event Count')
ax.set_title('Playback Events by Hour')
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

## Content Performance

In [ ]:
# Top content by views
spark.sql("""
    SELECT 
        content_id,
        COUNT(*) as total_events,
        SUM(CASE WHEN event_type = 'PLAY_START' THEN 1 ELSE 0 END) as play_starts,
        COUNT(DISTINCT user_id) as unique_viewers,
        ROUND(AVG(bitrate_kbps), 0) as avg_bitrate
    FROM iceberg.warehouse.playback_events
    GROUP BY content_id
    ORDER BY play_starts DESC
    LIMIT 10
""").show(truncate=False)

## Quality of Service Analysis

In [ ]:
# Rebuffer analysis
spark.sql("""
    SELECT 
        content_id,
        COUNT(*) as rebuffer_events,
        AVG(rebuffer_duration_ms) as avg_rebuffer_ms,
        MAX(rebuffer_duration_ms) as max_rebuffer_ms
    FROM iceberg.warehouse.playback_events
    WHERE event_type IN ('REBUFFER_START', 'REBUFFER_END')
    GROUP BY content_id
    HAVING COUNT(*) > 0
    ORDER BY rebuffer_events DESC
    LIMIT 10
""").show()

## Iceberg Table Metadata

In [ ]:
# View table snapshots (version history)
spark.sql("SELECT * FROM iceberg.warehouse.playback_events.snapshots").show(truncate=False)

In [ ]:
# View table files
files_df = spark.sql("""
    SELECT 
        file_path,
        file_format,
        record_count,
        file_size_in_bytes / 1024 / 1024 as size_mb
    FROM iceberg.warehouse.playback_events.files
""")

print(f"Total files: {files_df.count()}")
print(f"Total records: {files_df.agg({'record_count': 'sum'}).collect()[0][0]}")
print(f"Total size: {files_df.agg({'size_mb': 'sum'}).collect()[0][0]:.2f} MB")
files_df.show(10, truncate=False)

## Time Travel Query

In [ ]:
# Get available snapshots
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at 
    FROM iceberg.warehouse.playback_events.snapshots 
    ORDER BY committed_at DESC
""").collect()

if len(snapshots) > 1:
    old_snapshot_id = snapshots[-1].snapshot_id
    print(f"Querying old snapshot: {old_snapshot_id}")
    
    # Query an older version of the table
    spark.sql(f"""
        SELECT COUNT(*) as record_count
        FROM iceberg.warehouse.playback_events VERSION AS OF {old_snapshot_id}
    """).show()
else:
    print("Only one snapshot available")

## Cleanup

In [ ]:
spark.stop()
print("Session closed.")